# Hermite - Bogenelemente


In [1]:
import numpy 
import sympy 

phi, alpha, beta, R, E, I, A = sympy.symbols("varphi, alpha, beta, R, E, I, A ")

"""
HERMITE BOGENELEMENTE:


             ____R____(1)
            |          |
 BOGEN:     |          /
           R|         /                         KS:     __ x (i)
            |       /                                  |  
            | _ _ /                                    z
           (0)                                       (ii)

Abgebildeter Fall: alpha = pi/2

DoF_0: radiale translation......Knoten (0) 
DoF_1: rotation.................Knoten (0) 
DoF_2: radiale translation......Knoten (1) 
DoF_3: rotation.................Knoten (1) 
DoF_4: tangentiale translation..Knoten (0) 
DoF_5: tangentiale translation..Knoten (1) 

"""

def interpolationFunctions(x,l): 

        h = sympy.ones(6,1)
         
        h[0] =  1 - 3 * (x/l)**2 + 2 * (x/l)**3 
        h[1] =  - x * (1 - (x/l))**2
        h[2] =  3 * (x/l)**2 - 2 * (x/l)**3
        h[3] =  - x * ((x/l)**2 -(x/l))
        h[4] = 1 - x/l
        h[5] = x/l

        return h
    
h = interpolationFunctions(phi,alpha)  


from IPython.display import display
print("Ansatzfunktionen:")
print("-----------------")    
display(h)


Ansatzfunktionen:
-----------------


Matrix([
[1 - 3*varphi**2/alpha**2 + 2*varphi**3/alpha**3],
[                  -varphi*(1 - varphi/alpha)**2],
[    3*varphi**2/alpha**2 - 2*varphi**3/alpha**3],
[   -varphi*(-varphi/alpha + varphi**2/alpha**2)],
[                               1 - varphi/alpha],
[                                   varphi/alpha]])

In [2]:
def interpolationFunctionsDerivatives(h, phi):
        """ interpolationfunction[node, derivativeaxis] """
        dh_dX = sympy.ones(6, 2) 
        for fun in range(6): 
            dh_dX[fun,0] = sympy.diff(h[fun], phi) 
            dh_dX[fun,1] = sympy.diff(dh_dX[fun,0], phi) 
        return dh_dX 
    
dh_dx = interpolationFunctionsDerivatives(h, phi)

print("Ableitungen der Ansatzfunktionen:")
print("---------------------------------")    
display(dh_dx)

Ableitungen der Ansatzfunktionen:
---------------------------------


Matrix([
[                                 -6*varphi/alpha**2 + 6*varphi**2/alpha**3,               -6/alpha**2 + 12*varphi/alpha**3],
[                -(1 - varphi/alpha)**2 + 2*varphi*(1 - varphi/alpha)/alpha, 4*(1 - varphi/alpha)/alpha - 2*varphi/alpha**2],
[                                  6*varphi/alpha**2 - 6*varphi**2/alpha**3,                6/alpha**2 - 12*varphi/alpha**3],
[-varphi*(-1/alpha + 2*varphi/alpha**2) + varphi/alpha - varphi**2/alpha**2,                    2/alpha - 6*varphi/alpha**2],
[                                                                  -1/alpha,                                              0],
[                                                                   1/alpha,                                              0]])

In [3]:
U_ = numpy.asarray(sympy.symbols('U:' + str(6)))

w    = U_[0] *     h[0]   + U_[1] *     h[1]   + U_[2] *     h[2]   + U_[3] *     h[3]    # w
w_xx = U_[0] * dh_dx[0,1] + U_[1] * dh_dx[1,1] + U_[2] * dh_dx[2,1] + U_[3] * dh_dx[3,1]  # w''
u_x  = U_[4] * dh_dx[4,0] + U_[5] * dh_dx[5,0]                                            # u' 

""" Siehe PARKUS S.247 """
M = (E*I*( u_x - w_xx ))/(R**2)
N = (E*A*( w_xx + w + (M*R**2)/(E*I) ))/R


W = M**2/(E*I) + N**2/(E*A) # Ergänzungsenergiedichte

display(W)

A*E*(U0*(1 - 3*varphi**2/alpha**2 + 2*varphi**3/alpha**3) - U1*varphi*(1 - varphi/alpha)**2 + U2*(3*varphi**2/alpha**2 - 2*varphi**3/alpha**3) - U3*varphi*(-varphi/alpha + varphi**2/alpha**2) - U4/alpha + U5/alpha)**2/R**2 + E*I*(-U0*(-6/alpha**2 + 12*varphi/alpha**3) - U1*(4*(1 - varphi/alpha)/alpha - 2*varphi/alpha**2) - U2*(6/alpha**2 - 12*varphi/alpha**3) - U3*(2/alpha - 6*varphi/alpha**2) - U4/alpha + U5/alpha)**2/R**4

In [4]:
W = sympy.integrate( W*R, ( phi, 0, alpha ) ) # Ergänzungsenergie

In [5]:
esm = sympy.zeros(6,6)
for i in range(6): 
    for j in range(6): 
        esm[i,j] = sympy.diff( sympy.diff(W, U_[i]), U_[j] )

print(" ESM: ")
print("------")
display(esm)

 ESM: 
------


Matrix([
[             96*A*E*alpha/(35*R) - 144*E*I/(R**3*alpha**3) + (-12*A*E*R**2*alpha**4 + 288*E*I)/(3*R**3*alpha**3) + (2*A*E*R**2*alpha**4 + 72*E*I)/(R**3*alpha**3), -46*A*E*alpha**2/(105*R) - 48*E*I/(R**3*alpha**2) + (-A*E*R**2*alpha**5 + 84*E*I*alpha)/(R**3*alpha**3) + (4*A*E*R**2*alpha**5 - 144*E*I*alpha)/(3*R**3*alpha**3), -61*A*E*alpha/(35*R) + 72*E*I/(R**3*alpha**3) + (6*A*E*R**2*alpha**4 - 288*E*I)/(3*R**3*alpha**3), -127*A*E*alpha**2/(210*R) + 36*E*I/(R**3*alpha**2) + (2*A*E*R**2*alpha**5 - 144*E*I*alpha)/(3*R**3*alpha**3),         A*E/R + 12*E*I/(R**3*alpha**2) + (-2*A*E*R**2*alpha**3 - 12*E*I*alpha)/(R**3*alpha**3),         -A*E/R - 12*E*I/(R**3*alpha**2) + (2*A*E*R**2*alpha**3 + 12*E*I*alpha)/(R**3*alpha**3)],
[-46*A*E*alpha**2/(105*R) - 48*E*I/(R**3*alpha**2) + (-A*E*R**2*alpha**5 + 84*E*I*alpha)/(R**3*alpha**3) + (4*A*E*R**2*alpha**5 - 144*E*I*alpha)/(3*R**3*alpha**3),                                                        -68*A*E*alpha**3/(105*R) - 16*E*I/(R**3*alp

In [6]:
ESM = numpy.asarray(esm)
for index in numpy.ndindex(6,6): 
    print("k_e_local",index,"=",ESM[index])

k_e_local (0, 0) = 96*A*E*alpha/(35*R) - 144*E*I/(R**3*alpha**3) + (-12*A*E*R**2*alpha**4 + 288*E*I)/(3*R**3*alpha**3) + (2*A*E*R**2*alpha**4 + 72*E*I)/(R**3*alpha**3)
k_e_local (0, 1) = -46*A*E*alpha**2/(105*R) - 48*E*I/(R**3*alpha**2) + (-A*E*R**2*alpha**5 + 84*E*I*alpha)/(R**3*alpha**3) + (4*A*E*R**2*alpha**5 - 144*E*I*alpha)/(3*R**3*alpha**3)
k_e_local (0, 2) = -61*A*E*alpha/(35*R) + 72*E*I/(R**3*alpha**3) + (6*A*E*R**2*alpha**4 - 288*E*I)/(3*R**3*alpha**3)
k_e_local (0, 3) = -127*A*E*alpha**2/(210*R) + 36*E*I/(R**3*alpha**2) + (2*A*E*R**2*alpha**5 - 144*E*I*alpha)/(3*R**3*alpha**3)
k_e_local (0, 4) = A*E/R + 12*E*I/(R**3*alpha**2) + (-2*A*E*R**2*alpha**3 - 12*E*I*alpha)/(R**3*alpha**3)
k_e_local (0, 5) = -A*E/R - 12*E*I/(R**3*alpha**2) + (2*A*E*R**2*alpha**3 + 12*E*I*alpha)/(R**3*alpha**3)
k_e_local (1, 0) = -46*A*E*alpha**2/(105*R) - 48*E*I/(R**3*alpha**2) + (-A*E*R**2*alpha**5 + 84*E*I*alpha)/(R**3*alpha**3) + (4*A*E*R**2*alpha**5 - 144*E*I*alpha)/(3*R**3*alpha**3)
k_e_local (1,